In [6]:
import pandas as pd
import os, shutil

# Rutas guardadas
PROC = "../data/processed"
MODEL = "../data/model"

# Directorio de donde obendremos los datos procesados
clientes = pd.read_parquet(f"{PROC}/clientes.parquet")
polizas = pd.read_parquet(f"{PROC}/polizas.parquet")
siniestros = pd.read_parquet(f"{PROC}/siniestros.parquet")
pagos = pd.read_parquet(f"{PROC}/pagos.parquet")

print(clientes.shape, polizas.shape, siniestros.shape, pagos.shape)

(5000, 8) (8000, 7) (4000, 6) (20000, 5)


In [7]:
# Mapeo de valores de genero a un formato estandarizado
mapa_genero = { "m":"M", "masculino":"M", "f":"F", "femenino":"F" }

# Creamos una copia de la tabla de clientes y estandarizamos el genero
dim_clientes = clientes.copy()
dim_clientes["genero"] = dim_clientes["genero"].str.lower().str.strip().map(mapa_genero).fillna("Desconocido")

# Mostramos la cantidad de registros por genero, incluyendo los valores nulos
dim_clientes["genero"].value_counts(dropna=False)

genero
M              2147
F              2128
Desconocido     725
Name: count, dtype: int64

In [8]:
dim_producto = pd.DataFrame({ "producto" : sorted(polizas["producto"].unique().dropna()) })

dim_producto.insert(0,"producto_key", range(1, len(dim_producto) + 1))

dim_producto

,producto_key,producto
0,1,Auto
1,2,Gastos Médicos
2,3,Hogar
3,4,Vida


In [9]:
# Generamos la dimensión de fechas a partir de las fechas de inicio y fin de pólizas, fecha de siniestros y fecha de pagos
fechas = pd.concat([polizas["fecha_inicio"], polizas["fecha_fin"], siniestros["fecha_siniestro"], pagos["fecha_pago"]])

rango = pd.date_range(fechas.min(), fechas.max(), freq="D")

dim_fecha = pd.DataFrame({ "fecha" : rango})

dim_fecha

,fecha
0,2021-01-01
1,2021-01-02
2,2021-01-03
3,2021-01-04
4,2021-01-05
...,...
2430,2027-08-28
2431,2027-08-29
2432,2027-08-30
2433,2027-08-31


In [10]:
nombres_mes = {  
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

# Creamos una columna con la fecha en formato YYYYMMDD para usarla como llave primaria
dim_fecha["fecha_key"] = dim_fecha["fecha"].dt.strftime("%Y%m%d").astype(int)
dim_fecha["anio"] = dim_fecha["fecha"].dt.year
dim_fecha["mes"] = dim_fecha["fecha"].dt.month
dim_fecha["trimestre"] = dim_fecha["fecha"].dt.quarter
dim_fecha["nombre_mes"] = dim_fecha["mes"].map(nombres_mes)
dim_fecha["fecha"] = dim_fecha["fecha"].dt.date
dim_fecha = dim_fecha[["fecha_key", "fecha", "anio", "mes", "nombre_mes", "trimestre"]]

print(dim_fecha.head())


   fecha_key       fecha  anio  mes nombre_mes  trimestre
0   20210101  2021-01-01  2021    1      Enero          1
1   20210102  2021-01-02  2021    1      Enero          1
2   20210103  2021-01-03  2021    1      Enero          1
3   20210104  2021-01-04  2021    1      Enero          1
4   20210105  2021-01-05  2021    1      Enero          1


In [11]:
def fecha_key(serie):
    """Función para obtener la llave primaria de la dimensión de fechas a partir de una fecha en formato datetime"""
    return serie.dt.strftime("%Y%m%d").astype("Int64")

fact_polizas = polizas.merge(dim_producto, on="producto", how="left")
fact_polizas["fecha_inicio_key"] = fecha_key(fact_polizas["fecha_inicio"])
fact_polizas["fecha_fin_key"] = fecha_key(fact_polizas["fecha_fin"])
fact_polizas["anio"] = fact_polizas["fecha_inicio"].dt.year
fact_polizas = fact_polizas[["policy_id", "customer_id", "producto_key", "fecha_inicio_key", "fecha_fin_key", "estatus", "prima_mxn", "anio"]]

print(len(fact_polizas), len(polizas))
print(fact_polizas["producto_key"].isna().sum())
fact_polizas.head()



8000 8000
0


,policy_id,customer_id,producto_key,fecha_inicio_key,fecha_fin_key,estatus,prima_mxn,anio
0,P000001,C03781,3,20260825,20270825,Vencida,42465.37,2026
1,P000002,C00801,3,20240111,20250110,Activa,42661.10,2024
2,P000003,C03944,2,20220607,20230607,Activa,48509.71,2022
3,P000004,C01029,4,20230104,20240104,Activa,43618.35,2023
4,P000005,C04269,4,20220105,20230105,Cancelada,57649.37,2022


In [12]:
# Creamos la tabla de hechos de siniestros
pol_ref = polizas[["policy_id", "customer_id", "producto"]].merge(dim_producto, on="producto", how="left")[["policy_id", "customer_id", "producto_key"]]

fact_siniestros = siniestros.merge(pol_ref, on="policy_id", how="left")
fact_siniestros["fecha_siniestro_key"] = fecha_key(fact_siniestros["fecha_siniestro"])
fact_siniestros["anio"] = fact_siniestros["fecha_siniestro"].dt.year
fact_siniestros = fact_siniestros[["claim_id", "policy_id", "customer_id", "producto_key", "fecha_siniestro_key", "monto_mxn", "estatus", "categoria", "anio"]]

print(len(siniestros), len(fact_siniestros))
fact_siniestros.head()

4000 4000


,claim_id,policy_id,customer_id,producto_key,fecha_siniestro_key,monto_mxn,estatus,categoria,anio
0,CL000001,P003266,C00762,1,20250511,64580.88,Rechazado,Robo,2025
1,CL000002,P006989,C02471,2,20230901,486120.98,En revisión,Hospitalización,2023
2,CL000003,P006295,C01369,4,20240716,220679.05,Pagado,Fallecimiento,2024
3,CL000004,P000840,C03728,4,20240921,412062.71,Abierto,Daños a terceros,2024
4,CL000005,P003401,C04188,3,20251016,87625.99,Pagado,Incendio,2025


In [13]:
fact_pagos = pagos.merge(pol_ref, on="policy_id", how="left")

fact_pagos["fecha_pago_key"] = fecha_key(fact_pagos["fecha_pago"])
fact_pagos["anio"] = fact_pagos["fecha_pago"].dt.year
fact_pagos = fact_pagos[["payment_id", "policy_id", "customer_id", "producto_key", "fecha_pago_key", "monto_mxn", "metodo", "anio"]]

print(len(pagos), len(fact_pagos))
fact_pagos.head()

20000 20000


,payment_id,policy_id,customer_id,producto_key,fecha_pago_key,monto_mxn,metodo,anio
0,PAY0000001,P003089,C03551,3,20260313,561.14,Domiciliación,2026
1,PAY0000002,P006108,C03282,1,20230824,3632.53,Tarjeta,2023
2,PAY0000003,P003261,C00727,3,20240302,4252.31,Transferencia,2024
3,PAY0000004,P003956,C02656,1,20220916,5176.46,OXXO,2022
4,PAY0000005,P002178,C04230,3,20230202,1551.52,OXXO,2023


In [14]:
assert len(fact_polizas) == len(polizas), "La cantidad de registros en fact_polizas no coincide con la cantidad de registros en polizas"
assert len(fact_siniestros) == len(siniestros), "La cantidad de registros en fact_siniestros no coincide con la cantidad de registros en siniestros"
assert len(fact_pagos) == len(pagos), "La cantidad de registros en fact_pagos no coincide con la cantidad de registros en pagos"

for f in (fact_polizas, fact_siniestros, fact_pagos):
    assert f["customer_id"].isin(dim_clientes["customer_id"]).all(), "Existen registros en la tabla de hechos que no tienen un customer_id válido en la dimensión de clientes"
    assert f["producto_key"].isin(dim_producto["producto_key"]).all(), "Existen registros en la tabla de hechos que no tienen un producto_key válido en la dimensión de productos"

assert fact_polizas["fecha_inicio_key"].isin(dim_fecha["fecha_key"]).all(), "Existen registros en fact_polizas que no tienen un fecha_inicio_key válido en la dimensión de fechas"
assert fact_polizas["fecha_fin_key"].isin(dim_fecha["fecha_key"]).all(), "Existen registros en fact_polizas que no tienen un fecha_fin_key válido en la dimensión de fechas"
assert fact_siniestros["fecha_siniestro_key"].isin(dim_fecha["fecha_key"]).all(), "Existen registros en fact_siniestros que no tienen un fecha_siniestro_key válido en la dimensión de fechas"
assert fact_pagos["fecha_pago_key"].isin(dim_fecha["fecha_key"]).all(), "Existen registros en fact_pagos que no tienen un fecha_pago_key válido en la dimensión de fechas"

print("validaciones OK")

validaciones OK


In [15]:
dims = { "dim_clientes" : dim_clientes, "dim_producto" : dim_producto, "dim_fecha" : dim_fecha }
facts = { "fact_polizas" : fact_polizas, "fact_siniestros" : fact_siniestros, "fact_pagos" : fact_pagos }

for nombre, df in dims.items():
    shutil.rmtree(f"{MODEL}/{nombre}", ignore_errors=True)
    os.makedirs(f"{MODEL}/{nombre}", exist_ok=True)
    df.to_parquet(f"{MODEL}/{nombre}/{nombre}.parquet", index=False)

for nombre, df in facts.items():
    shutil.rmtree(f"{MODEL}/{nombre}", ignore_errors=True)
    df.to_parquet(f"{MODEL}/{nombre}", partition_cols=["anio"], index=False)

print("modelo exportado")
print(sorted(os.listdir(f"{MODEL}/fact_polizas")))
print(len(pd.read_parquet(f"{MODEL}/fact_pagos")))

modelo exportado
['anio=2021', 'anio=2022', 'anio=2023', 'anio=2024', 'anio=2025', 'anio=2026']
20000


In [16]:
fact_polizas.dtypes

policy_id               str
customer_id             str
producto_key          int64
fecha_inicio_key      Int64
fecha_fin_key         Int64
estatus                 str
prima_mxn           float64
anio                  int32
dtype: object